In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 1. Create a repeatable source fixture

booking = {
    "booking_id": "B001",
    "passenger_id": "P001",
    "booking_status": "CONFIRMED",
    "created_at": "2026-11-14T07:55:00Z",
}

booking_segment = {
    "booking_segment_id": "BS001",
    "booking_id": booking["booking_id"],
    "original_flight_leg_id": "FL100_20261115",
    "assigned_flight_leg_id": "FL100_20261115",
    "assignment_status": "ASSIGNED",
}

rebooked_segment = {
    **booking_segment,
    "assigned_flight_leg_id": "FL200_20261115",
}


def assignment_event(payload, version, effective_at, emitted_at):
    segment_id = payload["booking_segment_id"]
    return {
        "schema_version": 1,
        "source_system": "booking_system",
        "event_id": f"EVT_{segment_id}_V{version}",
        "entity_type": "booking_segment",
        "entity_id": segment_id,
        "entity_version": version,
        "effective_at": effective_at,
        "emitted_at": emitted_at,
        "payload": payload,
    }


initial_assignment_event = assignment_event(
    booking_segment, 1, "2026-11-14T08:00:00Z", "2026-11-14T08:01:00Z"
)
current_assignment_event = assignment_event(
    rebooked_segment, 2, "2026-11-14T10:00:00Z", "2026-11-14T10:01:00Z"
)

# V2 arrives first, V1 arrives late, then V2 is retried.
incoming_events = [
    current_assignment_event,
    initial_assignment_event,
    current_assignment_event,
]

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
# 2. Bronze: preserve every delivery as raw JSON

CATALOG = "airline_disruption"
SCHEMA = "dev"
namespace = f"`{CATALOG}`.`{SCHEMA}`"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")

bronze_table = f"{namespace}.v01_bronze_assignment_deliveries"
history_table = f"{namespace}.v01_silver_assignment_history"
current_table = f"{namespace}.v01_silver_assignment_current"

data = []

for i,event in enumerate(incoming_events, start=1):
    data.append((i,json.dumps(event)))

bronze = spark.createDataFrame(
    data,
    schema="delivery_no INT, raw_event STRING",
)

bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

In [0]:
from pyspark.sql import functions as F

schema = spark.range(1) \
    .select(F.schema_of_json(F.lit(data[0][1]))) \
    .first()[0]

print(schema)

In [0]:
# 3. Silver: parse the raw events and remove identical retries

event_schema = """
    schema_version INT,
    source_system STRING,
    event_id STRING,
    entity_type STRING,
    entity_id STRING,
    entity_version INT,
    effective_at STRING,
    emitted_at STRING,
    payload STRUCT<
        booking_segment_id: STRING,
        booking_id: STRING,
        original_flight_leg_id: STRING,
        assigned_flight_leg_id: STRING,
        assignment_status: STRING
    >
"""

events = (
    spark.table(bronze_table)
    .select(F.from_json("raw_event", event_schema).alias("event"))
    .select("event.*")
    .dropDuplicates()
)

entity_key = ["source_system", "entity_type", "entity_id"]
version_key = [*entity_key, "entity_version"]


def fail_if_conflict(conflicts, message):
    example = conflicts.take(1)
    if example:
        raise ValueError(f"{message}: {example[0].asDict()}")


# Identical retries are already gone. If an event ID still has
# multiple rows within its source, those rows disagree.
event_id_conflicts = (
    events.groupBy("source_system", "event_id")
    .count()
    .where(F.col("count") > 1)
)
fail_if_conflict(event_id_conflicts, "Conflicting event ID")

# Different event IDs can still disagree about the same entity version.
# Business contents here are the effective time and the snapshot payload.
version_conflicts = (
    events.groupBy(*version_key)
    .agg(
        F.countDistinct(F.struct("effective_at", "payload")).alias("variants")
    )
    .where(F.col("variants") > 1)
)
fail_if_conflict(version_conflicts, "Conflicting entity version")

In [0]:
# 4. Silver history: keep one row per valid entity version

# If different IDs describe the same business version, choose one
# consistently. This only happens after checking that they agree.
history_window = Window.partitionBy(*version_key).orderBy("event_id")

history = (
    events.withColumn("_rn", F.row_number().over(history_window))
    .where(F.col("_rn") == 1)
    .drop("_rn")
)

history.write.format("delta").mode("overwrite").saveAsTable(history_table)

In [0]:
# 5. Silver current state: keep the highest version per entity

current_window = Window.partitionBy(*entity_key).orderBy(
    F.col("entity_version").desc()
)

current = (
    spark.table(history_table)
    .withColumn("_rn", F.row_number().over(current_window))
    .where(F.col("_rn") == 1)
    .select(
        "source_system",
        "entity_id",
        "entity_version",
        "event_id",
        "effective_at",
        "payload.booking_id",
        "payload.original_flight_leg_id",
        "payload.assigned_flight_leg_id",
        "payload.assignment_status",
    )
)

current.write.format("delta").mode("overwrite").saveAsTable(current_table)

In [0]:
# 6. Verify the milestone

print("Bronze deliveries:", spark.table(bronze_table).count())
print("Silver history rows:", spark.table(history_table).count())
print("Silver current rows:", spark.table(current_table).count())

display(spark.table(current_table))

In [0]:
from pyspark.sql import functions as F

bronze_event  = spark.table(bronze_table).select(F.col("delivery_no"),F.col("raw_event"))

display(bronze_event)

In [0]:
silver_history  = spark.table(history_table).select("*")

display(silver_history)